# 02 · The Delta Rule: making hidden layers learn
NumPy for the model · matplotlib for graphs · Runs on Google Colab

1. The problem: hidden weights never update
2. What is a slope?
3. The slope of step is 0
4. Sigmoid: a "smooth step"
5. The delta rule (one neuron ➜ ★ worked example ➜ XOR data)
6. Experiments

## Data: XOR-type points (one noisy cluster per corner)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)
corners = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

def make_gate_data(gate, n=50, spread=0.15):
    X = np.vstack([c + rng.normal(0, spread, (n, 2)) for c in corners])
    y = np.repeat(gate, n)
    idx = rng.permutation(len(X))
    return X[idx], y[idx]

def split(X, y, test_ratio=0.2):
    n_test = int(len(X) * test_ratio)
    return X[n_test:], X[:n_test], y[n_test:], y[:n_test]

def add_x0(X):
    return np.hstack([-np.ones((len(X), 1)), X])                  # x0 = -1, so w0 is the threshold

X, y = make_gate_data([0, 1, 1, 0])
Xtr, Xte, ytr, yte = split(X, y)
Xtr0, Xte0 = add_x0(Xtr), add_x0(Xte)
print("Shape:", X.shape, "| class 0:", (y == 0).sum(), "| class 1:", (y == 1).sum())
print(f"train: {len(ytr)} rows, test: {len(yte)} rows")

In [ ]:
g = np.linspace(-0.6, 1.6, 200)
G1, G2 = np.meshgrid(g, g)
GRID = np.c_[G1.ravel(), G2.ravel()]

def plot_region(ax, predict, X, y, title):
    # predict: function taking points (n×2) and returning 0/1 for each
    ax.contourf(G1, G2, predict(GRID).reshape(G1.shape), levels=[-0.5, 0.5, 1.5], colors=["#f6d5d5", "#d5efd5"])
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="green", edgecolors="k", s=25)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="red", marker="X", edgecolors="k", s=25)
    ax.set(xlim=(-0.6, 1.6), ylim=(-0.6, 1.6), xlabel="x1", ylabel="x2", title=title, aspect="equal")

def draw_hidden_lines(ax, W1):
    for j in range(W1.shape[1]):
        w0, w1, w2 = W1[:, j]
        ax.plot(g, (w0 - w1 * g) / w2, "k--")
    ax.set(xlim=(-0.6, 1.6), ylim=(-0.6, 1.6))

fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_region(ax, lambda P: np.zeros(len(P)), Xtr, ytr, "XOR-type training data")
plt.show()

---
# Part 1: the problem — hidden weights never update
Step network 2 ➜ 2 ➜ 1 trained with the perceptron rule (same 4 functions as before).

In [ ]:
def step(s):
    return 1 if s >= 0 else 0

def forward_pass(x, w):
    s = x @ w
    return step(s), s

def compute_error(y, y_hat):
    return y - y_hat

def backward_pass(w, x, error, lr):
    return w + lr * error * x


def train(X, y, w, lr, epochs):
    for epoch in range(1, epochs + 1):
        mistakes = 0
        for x, target in zip(X, y):
            y_hat, s = forward_pass(x, w)
            error = compute_error(target, y_hat)
            w = backward_pass(w, x, error, lr)
            mistakes += int(error != 0)
        print(f"Epoch {epoch:3d} | mistakes {mistakes:3d}/{len(y)}")
    return w

In [ ]:
W_hidden = rng.uniform(-1, 1, (3, 2))                             # column j = hidden neuron j: [w0, w1, w2]

def hidden_layer(P):
    return (add_x0(P) @ W_hidden >= 0).astype(int)                # step on every hidden neuron

W_before = W_hidden.copy()
w_out = train(add_x0(hidden_layer(Xtr)), ytr, rng.uniform(-1, 1, 3), lr=0.1, epochs=10)

print("\nHidden weights before:\n", W_before)
print("Hidden weights after:\n", W_hidden)
print("Changed?", not np.array_equal(W_before, W_hidden))

step_net = lambda P: (add_x0(hidden_layer(P)) @ w_out >= 0).astype(int)
print(f"\nTest accuracy: {100 * (step_net(Xte) == yte).mean():.1f}%")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
plot_region(ax, step_net, Xtr, ytr, "Hidden lines (dashed) never move")
draw_hidden_lines(ax, W_hidden)
plt.show()

**Why didn't the hidden weights move?** We need one idea first: the **slope**.

---
# Part 2: what is a slope?

$$\text{slope} = \frac{\text{change in output}}{\text{change in input}}$$

In [ ]:
def slope(f, w, nudge=0.001):
    return (f(w + nudge) - f(w)) / nudge

curve = lambda w: (w - 3) ** 2                                    # a simple curve, lowest at w = 3

for w in [0, 1, 2, 3, 4, 5]:
    sl = slope(curve, w)
    advice = "increase w" if sl < -0.01 else "decrease w" if sl > 0.01 else "at the bottom"
    print(f"w = {w}: slope = ({curve(w + 0.001):.4f} - {curve(w):.4f}) / 0.001 = {sl:+7.3f} -> {advice}")

- **Sign** ➜ which way is uphill (move the other way)
- **Size** ➜ how steep
- **0** ➜ flat: nudging changes nothing, no direction

In [ ]:
ws = np.linspace(-0.5, 6.5, 200)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ws, curve(ws))
for w in [0.5, 3, 5]:
    sl = slope(curve, w)
    t = np.linspace(w - 0.8, w + 0.8, 10)
    axes[0].plot(t, curve(w) + sl * (t - w), "r-", lw=3)
    axes[0].annotate(f"slope {sl:+.1f}", (w, curve(w) + 1.2), ha="center")
axes[0].set(xlabel="w", ylabel="f(w)", title="Slope at three points")

w, lr = 0.0, 0.1
path = [w]
for i in range(15):
    w = w - lr * slope(curve, w)                                  # walk downhill
    path.append(w)
print("Walking downhill, w = w - 0.1 × slope:", np.round(path, 3))
axes[1].plot(ws, curve(ws))
axes[1].plot(path, [curve(p) for p in path], "ro-")
axes[1].set(xlabel="w", ylabel="f(w)", title="Following the slope downhill")
plt.tight_layout()
plt.show()

---
# Part 3: the slope of step is 0

In [ ]:
for s in [-2, -0.5, 0, 0.5, 2]:
    print(f"s = {s:+.1f}: step(s) = {step(s)}, step(s + 0.001) = {step(s + 0.001)} -> slope = {slope(step, s):.1f}")

To fix a hidden neuron, its share of the error is: output error × its output weight × **slope of its activation**.

In [ ]:
H_tr = add_x0(hidden_layer(Xtr))
i = next(k for k in range(len(ytr)) if step(H_tr[k] @ w_out) != ytr[k])   # a sample the network gets wrong
error = ytr[i] - step(H_tr[i] @ w_out)
s_hidden = add_x0(Xtr[i:i + 1])[0] @ W_hidden
print(f"Sample x = {np.round(Xtr[i], 2)}, target {ytr[i]}, network says {step(H_tr[i] @ w_out)} -> error {error:+d}")
for j in range(2):
    sl = slope(step, s_hidden[j])
    print(f"hidden {j + 1}: error {error:+d} × weight {w_out[j + 1]:+.3f} × slope {sl:.1f} = {error * w_out[j + 1] * sl + 0.0:.1f}")

That is why the hidden weights never moved. We need a step-like function **with** a slope.

---
# Part 4: sigmoid, a "smooth step"

$$\text{sigmoid}(s) = \frac{1}{1 + e^{-s}} \qquad \text{slope} = \text{sigmoid}(s)\,(1 - \text{sigmoid}(s))$$

In [ ]:
def sigmoid(s):
    return 1 / (1 + np.exp(-s))

def d_sigmoid(s):
    return sigmoid(s) * (1 - sigmoid(s))

for s in [-2, -0.5, 0, 0.5, 2]:
    print(f"s = {s:+.1f}: step {step(s)} (slope {slope(step, s):.1f}) | sigmoid {sigmoid(s):.3f} "
          f"(slope by nudging {slope(sigmoid, s):.4f}, by formula {d_sigmoid(s):.4f})")

zs = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(zs, [step(z) for z in zs], label="step")
axes[0].plot(zs, sigmoid(zs), label="sigmoid")
axes[0].set(xlabel="s", title="Same shape: 0 on the left, 1 on the right")
axes[0].legend()
axes[1].plot(zs, np.zeros_like(zs), label="slope of step")
axes[1].plot(zs, d_sigmoid(zs), label="slope of sigmoid")
axes[1].set(xlabel="s", title="Only sigmoid has a slope")
axes[1].legend()
plt.show()

---
# Part 5: the delta rule
## 5a. One neuron

Loss $= (y - \hat{y})^2$ with $\hat{y} = \text{sigmoid}(x \cdot w)$. Its slope for each weight is $-2\,(y - \hat{y})\,\text{sigmoid}'(s)\,x$.
Walking downhill gives the **delta rule** (2 folded into $\eta$):

$$\delta = (y - \hat{y}) \cdot \text{sigmoid}'(s) \qquad \Delta w = \eta \cdot \delta \cdot x$$

In [ ]:
x, target = Xtr0[0], ytr[0]
w = rng.uniform(-1, 1, 3)
loss = lambda w: (target - sigmoid(x @ w)) ** 2

print(f"x = {np.round(x, 3)}, target {target}, y_hat = sigmoid(x · w) = {sigmoid(x @ w):.4f}\n")
for j in range(3):
    nudge = np.zeros(3)
    nudge[j] = 0.001
    by_nudging = (loss(w + nudge) - loss(w)) / 0.001
    by_formula = -2 * (target - sigmoid(x @ w)) * d_sigmoid(x @ w) * x[j]
    print(f"w{j}: slope by nudging = {by_nudging:+.5f} | by formula = {by_formula:+.5f}")

---
## ★ Worked example: one small network on XOR, by hand

```
x1 ──W13=0.1──► H3 ──W35=0.3──┐
   ╲ W14=0.4                   ├──► O5 ──► y5
x2 ──W23=0.8──► H4 ──W45=0.9──┘
     W24=0.6
```
All neurons use sigmoid. Thresholds $W_{03}, W_{04}, W_{05}$ (input $-1$) start at 0. η = 0.5.

Backward pass: $\delta_5 = (t - y_5)\,y_5(1-y_5)$, $\;\delta_3 = \delta_5 W_{35}\,y_3(1-y_3)$, $\;\delta_4 = \delta_5 W_{45}\,y_4(1-y_4)$, then $W \leftarrow W + \eta \cdot \delta \cdot \text{input}$

In [ ]:
XOR_ROWS = [(0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 0)]              # x1, x2, target
START = {"W13": 0.1, "W14": 0.4, "W23": 0.8, "W24": 0.6, "W35": 0.3, "W45": 0.9,
         "W03": 0.0, "W04": 0.0, "W05": 0.0}                         # W0x = threshold of neuron x

def diagram_forward(x1, x2, W):
    y3 = sigmoid(x1 * W["W13"] + x2 * W["W23"] - W["W03"])
    y4 = sigmoid(x1 * W["W14"] + x2 * W["W24"] - W["W04"])
    y5 = sigmoid(y3 * W["W35"] + y4 * W["W45"] - W["W05"])
    return y3, y4, y5

def diagram_backward(x1, x2, t, W, lr, learn_thresholds):
    y3, y4, y5 = diagram_forward(x1, x2, W)
    d5 = (t - y5) * y5 * (1 - y5)
    d3 = d5 * W["W35"] * y3 * (1 - y3)
    d4 = d5 * W["W45"] * y4 * (1 - y4)
    inputs = {"W35": (d5, y3), "W45": (d5, y4), "W13": (d3, x1), "W23": (d3, x2), "W14": (d4, x1), "W24": (d4, x2)}
    if learn_thresholds:
        inputs.update({"W03": (d3, -1), "W04": (d4, -1), "W05": (d5, -1)})
    new_W = dict(W)
    for name, (d, inp) in inputs.items():
        new_W[name] = W[name] + lr * d * inp
    return new_W, (d5, d3, d4), inputs

In [ ]:
TEMP_ROWS = [(0.35, 0.9, 0.5)]

In [ ]:
W = dict(START)
for x1, x2, t in TEMP_ROWS:
    y3, y4, y5 = diagram_forward(x1, x2, W)
    print(f"x = ({x1},{x2}) | y3 = sigmoid({x1}×0.1 + {x2}×0.8) = {y3:.4f} | y4 = sigmoid({x1}×0.4 + {x2}×0.6) = {y4:.4f} | "
          f"y5 = sigmoid({y3:.4f}×0.3 + {y4:.4f}×0.9) = {y5:.4f} -> {int(y5 >= 0.5)} (target {t})")

In [ ]:
x1, x2, t = 0.35, 0.9, 0.5
y3, y4, y5 = diagram_forward(x1, x2, W)
new_W, (d5, d3, d4), inputs = diagram_backward(x1, x2, t, W, lr=0.5, learn_thresholds=True)

print(f"error = t - y5 = {t} - {y5:.4f} = {t - y5:+.4f}")
print(f"δ5 = error × y5(1 - y5)       = {t - y5:+.4f} × {y5 * (1 - y5):.4f} = {d5:+.5f}")
print(f"δ3 = δ5 × W35 × y3(1 - y3)    = {d5:+.5f} × 0.3 × {y3 * (1 - y3):.4f} = {d3:+.5f}")
print(f"δ4 = δ5 × W45 × y4(1 - y4)    = {d5:+.5f} × 0.9 × {y4 * (1 - y4):.4f} = {d4:+.5f}\n")
for name, (d, inp) in inputs.items():
    print(f"{name}: {W[name]:+.4f} + 0.5 × {d:+.5f} × {inp:+.4f} = {new_W[name]:+.4f}")

### Step 1: forward pass on all 4 XOR rows with the starting weights

In [ ]:
W = dict(START)
for x1, x2, t in XOR_ROWS:
    y3, y4, y5 = diagram_forward(x1, x2, W)
    print(f"x = ({x1},{x2}) | y3 = sigmoid({x1}×0.1 + {x2}×0.8) = {y3:.4f} | y4 = sigmoid({x1}×0.4 + {x2}×0.6) = {y4:.4f} | "
          f"y5 = sigmoid({y3:.4f}×0.3 + {y4:.4f}×0.9) = {y5:.4f} -> {int(y5 >= 0.5)} (target {t})")

### Step 2: one backward pass by hand on x = (1, 1), target 0

In [ ]:
x1, x2, t = 1, 1, 0
y3, y4, y5 = diagram_forward(x1, x2, W)
new_W, (d5, d3, d4), inputs = diagram_backward(x1, x2, t, W, lr=0.5, learn_thresholds=True)

print(f"error = t - y5 = {t} - {y5:.4f} = {t - y5:+.4f}")
print(f"δ5 = error × y5(1 - y5)       = {t - y5:+.4f} × {y5 * (1 - y5):.4f} = {d5:+.5f}")
print(f"δ3 = δ5 × W35 × y3(1 - y3)    = {d5:+.5f} × 0.3 × {y3 * (1 - y3):.4f} = {d3:+.5f}")
print(f"δ4 = δ5 × W45 × y4(1 - y4)    = {d5:+.5f} × 0.9 × {y4 * (1 - y4):.4f} = {d4:+.5f}\n")
for name, (d, inp) in inputs.items():
    print(f"{name}: {W[name]:+.4f} + 0.5 × {d:+.5f} × {inp:+.4f} = {new_W[name]:+.4f}")

In [ ]:
print("Repeat on the same row: y5 moves towards the target 0")
W_rep = dict(START)
for k in range(6):
    print(f"pass {k}: y5 = {diagram_forward(1, 1, W_rep)[2]:.4f}")
    W_rep, _, _ = diagram_backward(1, 1, 0, W_rep, lr=0.5, learn_thresholds=True)

### Step 3: train on all 4 rows — without thresholds vs with thresholds

In [ ]:
def diagram_train(learn_thresholds, lr=0.5, epochs=10000):
    W = dict(START)
    history = []
    for epoch in range(epochs):
        for x1, x2, t in XOR_ROWS:
            W, _, _ = diagram_backward(x1, x2, t, W, lr, learn_thresholds)
        history.append([diagram_forward(x1, x2, W)[2] for x1, x2, _ in XOR_ROWS])
    return W, np.array(history)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, learn in zip(axes, [False, True]):
    W_final, hist = diagram_train(learn)
    label = "with thresholds W03, W04, W05" if learn else "as drawn (no thresholds)"
    print(f"{label}:")
    for (x1, x2, t), out in zip(XOR_ROWS, hist[-1]):
        print(f"   x = ({x1},{x2}) -> y5 = {out:.3f} -> {int(out >= 0.5)} (target {t})")
    for k, (x1, x2, t) in enumerate(XOR_ROWS):
        ax.plot(hist[:, k], label=f"({x1},{x2}) target {t}")
    ax.axhline(0.5, color="gray", ls=":")
    ax.set(xlabel="epoch", ylabel="y5", title=label, ylim=(0, 1))
    ax.legend()
print("\nLearned weights with thresholds:", {k: round(float(v), 2) for k, v in W_final.items()})
plt.tight_layout()
plt.show()

Without thresholds, (1,1) can't get below 0.5. With thresholds, XOR is learned. Next: the same steps with matrices, random starting weights and the noisy XOR data.

---
## 5b. Hidden layer + delta rule on the XOR data (matrices)

| Function | What it does |
|---|---|
| `mlp_forward_pass(x, net)` | $h = \text{sigmoid}(x \cdot W_1)$, $\hat{y} = \text{sigmoid}(h \cdot W_2)$ |
| `compute_error(y, y_hat)` | $y - \hat{y}$ (same as before) |
| `mlp_backward_pass(net, x, error, cache, lr)` | $\delta_{out}$, then $\delta_{hidden} = \delta_{out} \cdot W_2 \cdot \text{sigmoid}'(s_1)$, then $W \leftarrow W - \eta \cdot dW$ |

In [ ]:
def new_net(hidden):
    return {"W1": rng.uniform(-1, 1, (3, hidden)),                # [w0, w1, w2] for every hidden neuron
            "W2": rng.uniform(-1, 1, hidden + 1)}                 # [w0, weight of every hidden neuron]

def mlp_forward_pass(x, net):
    s1 = x @ net["W1"]
    h = np.r_[-1, sigmoid(s1)]                                    # h0 = -1 for the output threshold
    s2 = h @ net["W2"]
    return sigmoid(s2), (s1, h, s2)

def mlp_backward_pass(net, x, error, cache, lr):
    s1, h, s2 = cache
    delta_out = error * d_sigmoid(s2)
    delta_hidden = delta_out * net["W2"][1:] * d_sigmoid(s1)      # error passed back through the slope
    dW2 = -delta_out * h                                          # gradient of the loss
    dW1 = -np.outer(x, delta_hidden)
    return {"W1": net["W1"] - lr * dW1, "W2": net["W2"] - lr * dW2}

def mlp_predict(P, net):
    H = np.hstack([-np.ones((len(P), 1)), sigmoid(add_x0(P) @ net["W1"])])
    return (sigmoid(H @ net["W2"]) >= 0.5).astype(int)

def mlp_accuracy(X0, y, net):
    return 100 * (mlp_predict(X0[:, 1:], net) == y).mean()


def mlp_train(X, y, net, lr, epochs, show=True):
    losses, snapshots = [], {}
    for epoch in range(1, epochs + 1):
        errors = []
        for x, target in zip(X, y):
            y_hat, cache = mlp_forward_pass(x, net)
            error = compute_error(target, y_hat)
            net = mlp_backward_pass(net, x, error, cache, lr)
            errors.append(error)
        losses.append(np.mean(np.square(errors)))
        snapshots[epoch] = net
        if show and (epoch == 1 or epoch % (epochs // 10) == 0):
            print(f"Epoch {epoch:4d} | loss {losses[-1]:.4f} | train accuracy {mlp_accuracy(X, y, net):5.1f}%")
    return net, losses, snapshots

In [ ]:
net, losses, snapshots = mlp_train(Xtr0, ytr, new_net(hidden=2), lr=0.5, epochs=200)
print(f"\nTest accuracy: {mlp_accuracy(Xte0, yte, net):.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses)
ax.set(xlabel="epoch", ylabel="loss (mean error²)", title="Loss per epoch")
plt.show()

epochs_to_show = [1, 20, 50, 200]
fig, axes = plt.subplots(1, len(epochs_to_show) + 1, figsize=(22, 4.5))
for ax, e in zip(axes, epochs_to_show):
    plot_region(ax, lambda P: mlp_predict(P, snapshots[e]), Xtr, ytr,
                f"after epoch {e}: {mlp_accuracy(Xtr0, ytr, snapshots[e]):.0f}%")
plot_region(axes[-1], lambda P: mlp_predict(P, net), Xte, yte, f"Test: {mlp_accuracy(Xte0, yte, net):.0f}% (dashed = hidden lines)")
draw_hidden_lines(axes[-1], net["W1"])
plt.tight_layout()
plt.show()

---
# Part 6: Experiments
## 1. Learning rate (same start)

In [ ]:
start = new_net(2)
fig, ax = plt.subplots(figsize=(8, 3.5))
print(f"{'η':>6} | {'test acc':>8}")
print("-" * 18)
for lr in [0.01, 0.1, 0.5, 2, 10, 50]:
    trained, loss_run, _ = mlp_train(Xtr0, ytr, start, lr, 100, show=False)
    print(f"{lr:>6} | {mlp_accuracy(Xte0, yte, trained):7.1f}%")
    ax.plot(loss_run, label=f"η = {lr}")
ax.set(xlabel="epoch", ylabel="loss", title="Too small is slow, too big jumps around")
ax.legend()
plt.show()

## 2. Hidden layer size (η = 0.5, 20 random starts)

In [ ]:
print(f"{'hidden':>6} | {'solved (test ≥ 95%)':>19}")
print("-" * 29)
for hidden in [1, 2, 4, 8]:
    solved = sum(mlp_accuracy(Xte0, yte, mlp_train(Xtr0, ytr, new_net(hidden), 0.5, 100, show=False)[0]) >= 95
                 for _ in range(20))
    print(f"{hidden:>6} | {solved:>13}/20")

Next notebook: other activation functions (tanh, ReLU, …) and their advantages and limitations.